In [1]:
import pandas as pd
import altair as alt

df = pd.read_csv("medicare_with_conditions.csv")

conditions = ["Hypertension", "High Cholesterol", "Diabetes", "Obesity", "Arthritis"]
claims = ["Tot_Clms_2019", "Tot_Clms_2020", "Tot_Clms_2021", "Tot_Clms_2022", "Tot_Clms_2023"]

# create a new df with only the top 5 conditions we are looking at and organize the claims by year
conditions_df = df[df['Disease_Category'].isin(conditions)]
new_df = conditions_df.melt(id_vars = ["Disease_Category"], 
                            value_vars = claims, 
                            var_name = "Year", 
                            value_name = "Total_Claims")

new_df["Year"] = new_df["Year"].str[-4:].astype(int)

# find the total number of claims per year for each condition
sum_df = new_df.groupby(["Disease_Category", "Year"], as_index = False)["Total_Claims"].sum()

# create line plot using altair
input_dropdown = alt.binding_select(options = [None] + conditions,
                                    labels = ["All"] + conditions,
                                    name = "Chronic Condition: ")
selection = alt.selection_point(fields = ["Disease_Category"], bind = input_dropdown)

chart = alt.Chart(sum_df).mark_line(point = True).encode(
    x = alt.X("Year:O"),
    y = alt.Y("Total_Claims:Q", title = "Total Claims"),
    color = alt.Color("Disease_Category:N", legend = alt.Legend(title="Chronic Condition")),
    tooltip = [alt.Tooltip("Disease_Category:N", title="Condition"),
               alt.Tooltip("Year:O"),
               alt.Tooltip("Total_Claims:Q", title="Total Claims")]
).add_params(
    selection
).transform_filter(
    selection
).properties(
    width = 600,
    height = 400,
    title = "Total Claims From 2019 to 2023 for Selected Condition"
)

In [2]:
chart.save('altair.html')